# Tests des fonctions de calcul de la saturation - full chunk

découpage 1 : à partir des pools
découpage 2 : à partir des stations (après élimination des stations déja traitées)

In [44]:
import json
from datetime import date, timedelta

import pandas as pd
from e2_e3_e6 import (
    e2,
    e3,
    e6,
    # filter_statuses_sessions,
    get_chunked_state_grp,
    get_chunked_state_poc,
    get_chunked_state_pools,
    # get_sampled_state_poc,
    get_state_poc_for_chunk,
)
from pandas import NamedAgg

#from saturation_image_quali_prod import (
from utils import (
    filter_sessions_duration,
    to_sampled_state_grp,
    to_state_grp,
    # to_state_poc,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,8, 1)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

In [3]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("../source/tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics

In [4]:
e1_statics = read_statics_pools(day, MIN_POWER)
e1_statics

,id_pool,id_station_itinerance
0,A000001,FRHPCPNF080371TIERSTOTEM
13,A000002,FRTSLP5670
14,A000002,FRIOYP13531046
29,A000003,FRIOYP13530804
51,A000004,FRFASP11568703
...,...,...
7112,A001871,NaN
7113,A001872,NaN
7114,A001873,NaN
7115,A001874,NaN


## test qualicharge

In [ ]:
from datetime import datetime
    
samples_per_day = SAMPLES
min_power = MIN_POWER
chunk_size = 200

min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)

statuses = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
sessions = filter_sessions_duration(
    sessions_s3, min_duration=min_duration, max_duration=max_duration
)
statics = read_statics(day, MIN_POWER)
statics = statics[statics["puissance_nominale"] >= min_power]

sessions_poc = (
    sessions.groupby(ID_POC)
    .agg(
        sessions_nb=NamedAgg("energy", "count"),
        energy_cum=NamedAgg("energy", "sum"),
    )
    .reset_index()
)
#pools_statics = get_station_pool_for_day(day, environment)
pools_statics = read_statics_pools(day, MIN_POWER)
pools_stations = pools_statics[[ID_POOL, ID_STATION]].drop_duplicates()
pools_stations_pocs = pools_statics.merge(statics, on=ID_STATION, how="left")

In [73]:
def get_groups(statics, id_grp, chunk_size):
    """Group pdc based on cumulative sum of nb_pdc."""
    df_nb_pocs = statics.groupby(id_grp).agg(nb_pocs=NamedAgg(ID_POC, "count")).reset_index()
    df_nb_pocs["cumsum"] = df_nb_pocs["nb_pocs"].cumsum()
    df_nb_pocs["group"] = (df_nb_pocs["cumsum"] - 1) // chunk_size
    field_pool = [ID_POOL] if id_grp == ID_POOL else []
    static_chunks = [
        statics[statics[id_grp].isin(group[id_grp])].copy()[[ID_POC, ID_STATION] + field_pool]
        for _, group in df_nb_pocs.groupby("group")
    ]
    return static_chunks

In [ ]:
#chunks = get_groups(pools_stations_pocs, chunk_size)
#chunks[0][ID_POC].dropna()

In [ ]:
def to_state(statics: pd.DataFrame, 
             chunk: pd.DataFrame, 
             sessions: pd.DataFrame, 
             statuses: pd.DataFrame, 
             day: date, 
             samples_per_day: int,
             add_pool: bool = True,
             ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Convert the data to a state representation."""
    statics_chunk = statics[statics[ID_POC].isin(chunk[ID_POC])]
    sampled_state_poc, state_poc = get_state_poc_for_chunk(
        day, samples_per_day, statics_chunk, sessions, statuses
    )
    state_station = to_state_grp(
        to_sampled_state_grp(
            sampled_state_poc[sampled_state_poc[ID_POC].isin(chunk[ID_POC])],
            chunk[[ID_POC, ID_STATION]],
            ID_STATION,
            SATURATION_RATIO,
            OVERLOAD_RATIO,
            add_full_use=True,
            add_latency=True,
        ),  # type: ignore[call-overload]
        ID_STATION,
        samples_per_day,
    )
    state_pool = to_state_grp(
        to_sampled_state_grp(
            sampled_state_poc[sampled_state_poc[ID_POC].isin(chunk[ID_POC])],
            chunk[[ID_POC, ID_POOL]],
            ID_POOL,
            SATURATION_RATIO,
            OVERLOAD_RATIO,
            add_full_use=True,
            add_latency=True,
        ),  # type: ignore[call-overload]
        ID_POOL,
        samples_per_day,
    ) if add_pool else pd.DataFrame()
    return (state_poc, state_station, state_pool)

In [ ]:
# chunk calculation for pools and stations
chunks_pools = get_groups(pools_stations_pocs, ID_POOL, chunk_size)
futures_pools = [
    to_state(# .submit(
        statics, chunk, sessions, statuses, day, samples_per_day
    ) 
    for chunk in chunks_pools
]
statics_stations = statics[~statics[ID_POC].isin(pools_stations_pocs[ID_POC])]
chunks_stations = get_groups(statics_stations, ID_STATION, chunk_size)
futures_stations = [
    to_state(# .submit(
        statics_stations, chunk, sessions, statuses, day, samples_per_day, add_pool=False
    ) 
    for chunk in chunks_stations
]

In [79]:
# e2 indicator
state_poc = pd.concat(
    [
        pd.concat([future[0] for future in futures_pools], ignore_index=True),
        pd.concat([future[0] for future in futures_stations], ignore_index=True),
    ],
    ignore_index=True,
)
indicators_e2 = e2(
    #environment,
    state_poc,
    sessions_poc,
    day,
    #create_artifact,
    #persist,
)

In [80]:
# e3 indicator
state_station = pd.concat(
    [
        pd.concat([future[1] for future in futures_pools], ignore_index=True),
        pd.concat([future[1] for future in futures_stations], ignore_index=True),
    ],
    ignore_index=True
)
sessions_stations = pd.merge(
    statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how="left"
).fillna(0)
info_sessions_stations = (
    sessions_stations[[ID_STATION, "sessions_nb", "energy_cum"]]
    .groupby(ID_STATION)
    .sum()
    .reset_index()
)
indicators_e3 = e3(
    #environment,
    state_station,
    info_sessions_stations,
    day,
    #create_artifact,
    #persist,
)

In [81]:
# e6 indicator
state_pool = pd.concat([future[2] for future in futures_pools], ignore_index=True)
sessions_pools = pd.merge(
    pools_stations_pocs[[ID_POOL, ID_POC]], sessions_poc, on=ID_POC, how="left"
).fillna(0)
info_sessions_pools = (
    sessions_pools[[ID_POOL, "sessions_nb", "energy_cum"]]
    .groupby(ID_POOL)
    .sum()
    .reset_index()
)
indicators_e6 = e6(
    #environment,
    state_pool,
    info_sessions_pools,
    day,
    #create_artifact,
    #persist,
)


In [70]:
state_poc.sort_values(by=[ID_POC])

,id_pdc_itinerance,occupe,occupe_max,hors_service,libre,pseudo_libre,pseudo_occupe
4135,FR3R3E10001456611,70.0,60.0,0.0,1370.0,70.0,0.0
4136,FR3R3E10001456612,80.0,45.0,0.0,1360.0,80.0,0.0
4137,FRALDE100541,145.0,55.0,0.0,1295.0,0.0,0.0
4138,FRALDE100551,170.0,60.0,0.0,1270.0,0.0,15.0
4139,FRALDE100552,90.0,45.0,0.0,1350.0,0.0,0.0
...,...,...,...,...,...,...,...
23346,FRZUNEFR8601ER06,220.0,55.0,15.0,1205.0,0.0,5.0
23347,FRZUNEFR8801ER01,175.0,60.0,0.0,1265.0,0.0,0.0
23348,FRZUNEFR8801ER02,140.0,55.0,0.0,1300.0,0.0,0.0
23349,FRZUNEFR8801ER03,65.0,35.0,0.0,1375.0,0.0,5.0


In [71]:
state_station.sort_values(by=[ID_STATION])

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
505,FR3R3P89882136,2,0.0,1290.0,0.0,0.0,0.0,150.0,0.0,0.0,0.0
506,FRALDPFR00916,8,0.0,1220.0,0.0,0.0,0.0,220.0,0.0,0.0,0.0
507,FRALDPFR00950,8,0.0,995.0,0.0,0.0,5.0,440.0,0.0,0.0,0.0
508,FRALLPGO000007,6,0.0,830.0,340.0,205.0,170.0,235.0,60.0,60.0,230.0
509,FRALLPGO000013,10,0.0,395.0,0.0,0.0,40.0,1005.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
5479,FRZUNP6023950095875504781,8,0.0,570.0,120.0,25.0,60.0,785.0,15.0,55.0,55.0
5480,FRZUNP6927750076048540479,8,0.0,1090.0,0.0,0.0,0.0,350.0,0.0,0.0,0.0
384,FRZUNP7329346578064027187,25,0.0,175.0,0.0,0.0,0.0,1265.0,0.0,0.0,0.0
5481,FRZUNP8610050047391683219,6,0.0,1000.0,235.0,20.0,115.0,305.0,20.0,60.0,150.0


In [72]:
state_pool.sort_values(by=[ID_POOL])

,id_pool,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
0,A000001,11,0.0,660.0,0.0,0.0,0.0,780.0,0.0,0.0,0.0
1,A000002,16,0.0,170.0,10.0,0.0,40.0,1230.0,0.0,10.0,10.0
2,A000003,16,0.0,385.0,25.0,10.0,30.0,1015.0,5.0,25.0,25.0
3,A000004,8,0.0,320.0,330.0,90.0,155.0,875.0,35.0,60.0,330.0
4,A000005,8,0.0,585.0,120.0,25.0,75.0,755.0,15.0,50.0,45.0
...,...,...,...,...,...,...,...,...,...,...,...
471,A001679,10,0.0,1000.0,0.0,0.0,0.0,440.0,0.0,0.0,0.0
472,A001685,6,0.0,55.0,810.0,375.0,270.0,740.0,45.0,60.0,420.0
473,A001698,12,0.0,455.0,320.0,170.0,120.0,695.0,50.0,60.0,310.0
474,A001701,2,0.0,1145.0,10.0,0.0,0.0,295.0,0.0,10.0,10.0


In [50]:
state_poc_station_pool = []
chunks = get_groups(pools_stations_pocs, group_size=200)
#chunks[0]
for chunk in chunks[0:1]:
    #print(chunk)
    statics_chunk = statics[statics[ID_POC].isin(chunk[ID_POC])]
    sampled_state_poc, state_poc = get_state_poc_for_chunk(
        day, samples_per_day, statics_chunk, sessions, statuses
    )
    state_station = to_state_grp(  # .submit(
                        to_sampled_state_grp(
                            sampled_state_poc[sampled_state_poc[ID_POC].isin(chunk[ID_POC])],
                            chunk[[ID_POC, ID_STATION]],
                            ID_STATION,
                            SATURATION_RATIO,
                            OVERLOAD_RATIO,
                            add_full_use=True,
                            add_latency=True,
                        ),  # type: ignore[call-overload]
                        ID_STATION,
                        samples_per_day,
                    )
    state_pool = to_state_grp(  # .submit(
                        to_sampled_state_grp(
                            sampled_state_poc[sampled_state_poc[ID_POC].isin(chunk[ID_POC])],
                            chunk[[ID_POC, ID_POOL]],
                            ID_POOL,
                            SATURATION_RATIO,
                            OVERLOAD_RATIO,
                            add_full_use=True,
                            add_latency=True,
                        ),  # type: ignore[call-overload]
                        ID_POOL,
                        samples_per_day,
                    )
    state_poc_station_pool.append((state_poc, state_station, state_pool))


In [7]:
len(statics), len(statics[ID_STATION].drop_duplicates())


(28654, 6166)

In [76]:
df_pdc = pd.DataFrame({
    "id_pdc_itinerance": ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8", "p9", "p10", "p11"],
    "id_station_itinerance": ["s1", "s1", "s21", "s22", "s3", "s3", "s4", "s5", "s5", "s6", "s6"],
    "id_pool": ["A1", "A1", "A2", "A2", "A3", "A3", "A4", "A5", "A5", "A6", "A6"]
})
df_nb_pdc = pd.DataFrame({
    "nb_pdc": [50, 80, 40, 70, 100, 20, 60, 30, 15, 5, 20],
    "id_pool": ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "A10", "A11"]
})

def get_groups(df_pdc, group_size):
    """Group pdc based on cumulative sum of nb_pdc."""
    df_nb_pdc = df_pdc.groupby(ID_POOL).agg(nb_pdc=NamedAgg(ID_POC, "count")).reset_index()
    df_nb_pdc["cumsum"] = df_nb_pdc["nb_pdc"].cumsum()
    df_nb_pdc["group"] = (df_nb_pdc["cumsum"] - 1) // group_size
    df_group = [
        df_pdc[df_pdc[ID_POOL].isin(group[ID_POOL])].copy()[[ID_POC, ID_STATION, ID_POOL]]
        for name, group in df_nb_pdc.groupby("group")
    ]
    return df_group

list_groups = get_groups(df_pdc, group_size=3)

assert list_groups[0][ID_POC].tolist() == ["p1", "p2"]
assert list_groups[0][ID_POOL].drop_duplicates().tolist() == ["A1"]
assert list_groups[1][ID_POC].tolist() == ["p3", "p4", "p5", "p6"]
assert list_groups[2][ID_POC].tolist() == ["p7", "p8","p9"]
assert list_groups[3][ID_POC].tolist() == ["p10", "p11"]